## Analysis 3: Product Distribution by Category

### Objective

The objective of this analysis is to examine the distribution of products across different categories using MongoDB aggregation queries. The `$lookup` stage is used to combine product data with category details, while `$group` and `$sort` are used to calculate and display the number of products in each category.

In [14]:
from pymongo import MongoClient
from dotenv import load_dotenv
import os

In [15]:
load_dotenv()

True

In [16]:
MONGODB_URI = os.getenv("MONGODB_URI")

if not MONGODB_URI:
    raise ValueError("MONGODB_URI is missing from the .env file")

In [17]:
client = MongoClient(MONGODB_URI)

In [18]:
db = client["ecommerce_analysis"]
products_collection = db["products"]

print("Successfully connected to MongoDB Atlas!")
print("Database:", db.name)
print("Collection:", products_collection.name)

Successfully connected to MongoDB Atlas!
Database: ecommerce_analysis
Collection: products


### Preview Products with Category Details

Use `$lookup` to match products with their category records, then inspect a sample of the joined documents.

In [19]:
pipeline = [
    {
        "$lookup": {
            "from": "categories",
            "localField": "category_id",
            "foreignField": "id",
            "as": "category_details"
        }
    },
    {
        "$unwind": "$category_details"
    },
    {
        "$project": {
            "_id": 0,
            "product_id": "$id",
            "product_name": "$name",
            "category_id": "$category_id",
            "category_details": "$category_details"
        }
    },
    {
        "$limit": 10
    }
]

for document in products_collection.aggregate(pipeline):
    print(document)

{'product_id': 1, 'product_name': 'Licensed Wooden Cheese', 'category_id': 20, 'category_details': {'_id': ObjectId('6ae73d62e04f6c5e941e7510'), 'id': 20, 'description': 'Free shipping over $35', 'name': 'Garden', 'parent_category_id': None, 'slug': 'garden'}}
{'product_id': 1, 'product_name': 'Licensed Wooden Cheese', 'category_id': 20, 'category_details': {'_id': ObjectId('6ae7478b5f11853766053b7e'), 'id': 20, 'description': 'Free shipping over $35', 'name': 'Garden', 'parent_category_id': None, 'slug': 'garden'}}
{'product_id': 2, 'product_name': 'Handmade Bronze Shirt', 'category_id': 11, 'category_details': {'_id': ObjectId('6ae73d62e04f6c5e941e7507'), 'id': 11, 'description': 'Bestsellers', 'name': 'Outdoors', 'parent_category_id': None, 'slug': 'outdoors'}}
{'product_id': 2, 'product_name': 'Handmade Bronze Shirt', 'category_id': 11, 'category_details': {'_id': ObjectId('6ae7478b5f11853766053b75'), 'id': 11, 'description': 'Bestsellers', 'name': 'Outdoors', 'parent_category_id':

### Count Products by Category ID

Group products by `category_id` and sort the results by product count to identify the largest categories.

In [20]:
pipeline = [
    {
        "$group": {
            "_id": "$category_id",
            "total_products": {"$sum": 1}
        }
    },
    {
        "$sort": {
            "total_products": -1
        }
    }
]

category_results = list(products_collection.aggregate(pipeline))

for result in category_results:
    print(result)

{'_id': 1, 'total_products': 166}
{'_id': 2, 'total_products': 74}
{'_id': 4, 'total_products': 54}
{'_id': 3, 'total_products': 48}
{'_id': 8, 'total_products': 44}
{'_id': 11, 'total_products': 34}
{'_id': 16, 'total_products': 30}
{'_id': 9, 'total_products': 30}
{'_id': 18, 'total_products': 30}
{'_id': 7, 'total_products': 30}
{'_id': 19, 'total_products': 30}
{'_id': 17, 'total_products': 28}
{'_id': 13, 'total_products': 24}
{'_id': 20, 'total_products': 24}
{'_id': 5, 'total_products': 24}
{'_id': 10, 'total_products': 22}
{'_id': 6, 'total_products': 22}
{'_id': 22, 'total_products': 20}
{'_id': 12, 'total_products': 20}
{'_id': 21, 'total_products': 18}
{'_id': 14, 'total_products': 14}
{'_id': 15, 'total_products': 14}


### Count Products by Category Name

Join category names to the products, then group and rank the results using readable category names.

In [21]:
pipeline = [
    {
        "$lookup": {
            "from": "categories",
            "localField": "category_id",
            "foreignField": "id",
            "as": "category_details"
        }
    },
    {
        "$unwind": "$category_details"
    },
    {
        "$group": {
            "_id": "$category_details.name",
            "total_products": {"$sum": 1}
        }
    },
    {
        "$sort": {
            "total_products": -1
        }
    }
]

category_results = list(products_collection.aggregate(pipeline))

for result in category_results:
    print(result)

{'_id': 'Games', 'total_products': 332}
{'_id': 'Grocery', 'total_products': 148}
{'_id': 'Beauty', 'total_products': 108}
{'_id': 'Computers', 'total_products': 96}
{'_id': 'Toys', 'total_products': 88}
{'_id': 'Outdoors', 'total_products': 68}
{'_id': 'Home', 'total_products': 60}
{'_id': 'Electronics', 'total_products': 60}
{'_id': 'Sports', 'total_products': 60}
{'_id': 'Movies', 'total_products': 60}
{'_id': 'Clothing', 'total_products': 60}
{'_id': 'Kids', 'total_products': 56}
{'_id': 'Jewelry', 'total_products': 48}
{'_id': 'Office', 'total_products': 48}
{'_id': 'Garden', 'total_products': 48}
{'_id': 'Tools', 'total_products': 44}
{'_id': 'Baby', 'total_products': 44}
{'_id': 'Books', 'total_products': 40}
{'_id': 'Shoes', 'total_products': 40}
{'_id': 'Automotive', 'total_products': 36}
{'_id': 'Music', 'total_products': 28}
{'_id': 'Health', 'total_products': 28}
